# Three-Agent Research System

This notebook implements the **Orchestrator-Worker pattern with Fan-out/Fan-in execution and a verification loop** using the [Strands Agents SDK](https://strandsagents.com) and Amazon Bedrock.

## Architecture

```
         [User Query]
               │
        ┌──────▼──────┐
        │   Dispatcher │  (entry point — formats the query for workers)
        └──────┬──────┘
      ┌────────┼────────┐
      ▼        ▼        ▼
 [Claude]  [Nova]  [Mistral]   ← Fan-out: all three run in parallel
      └────────┼────────┘
               │             ← Fan-in: waits for ALL three (AND semantics)
        ┌──────▼──────┐
        │  Aggregator  │  (finds consensus, disagreements, unique findings)
        └──────┬──────┘
         ┌─────┴──────┐
         ▼            ▼
     [Consensus]  [Verifier]  ← conditional: disputed claims → extra scrutiny
                      │
               ┌──────▼──────┐
               │ Synthesizer  │  (confidence-weighted final answer)
               └─────────────┘
```

### Why three different models?

**Council Mode consensus** leverages *model diversity as a feature*. Each model has different training data, RLHF tuning, and architectural biases. When all three agree, confidence is high. Disagreement is a signal — not noise — that warrants further investigation. Research shows this approach reduces hallucinations by ~36% compared to single-model responses.

### Models used (all via Amazon Bedrock us-east-1 inference profiles)

| Role | Model ID | Why |
|------|----------|-----|
| Worker 1 | `us.anthropic.claude-opus-4-7` | Strong reasoning, best for nuanced analysis |
| Worker 2 | `us.amazon.nova-pro-v1:0` | Amazon's flagship — different training lineage |
| Worker 3 | `openai.mistral.mistral-large-3-675b-instruct` | Mistral Large 3 — strong reasoning, different training lineage |
| Orchestrator | `us.anthropic.claude-opus-4-7` | Best at structured synthesis and reasoning |

> **Why `us.` prefix?** The `us.` prefix identifies a US cross-region inference profile — Bedrock routes your request across `us-east-1`, `us-east-2`, and `us-west-2` for availability. Several models (all Mistral, some newer models) *require* inference profiles and reject plain on-demand calls. Using `us.*` IDs consistently is the safe default.

## 1. Installation

Install the Strands Agents SDK and its Bedrock dependency. The `strands-agents-tools` package adds optional built-in tools (web search, file I/O, etc.) — not required for this notebook but useful for future extension.

In [1]:
%pip install strands-agents strands-agents-tools boto3 python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2. AWS Credentials

Bedrock uses standard boto3 credential resolution in this order:
1. Environment variables (`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`)
2. `~/.aws/credentials` file
3. IAM instance role (if running on EC2/ECS)

The cell below loads from a `.env` file if present, then verifies credentials are accessible. **You do not need to hard-code keys anywhere.**

In [2]:
import os
from pathlib import Path

# Load .env if it exists (copy .env.example → .env and fill in your values)
env_file = Path(".env")
if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded credentials from .env")

# Verify boto3 can resolve credentials
import boto3
try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"AWS Account : {identity['Account']}")
    print(f"ARN         : {identity['Arn']}")
    print(f"Region      : {boto3.Session().region_name or os.getenv('AWS_DEFAULT_REGION', 'not set')}")
except Exception as e:
    print(f"Credential check failed: {e}")
    print("Set AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_DEFAULT_REGION in your environment or .env file.")

Loaded credentials from .env
AWS Account : 744387643533
ARN         : arn:aws:iam::744387643533:user/byungdc+cli
Region      : us-west-2


## 3. Imports

Key components from Strands:
- `Agent` — the core agent class; wraps a model and a system prompt
- `BedrockModel` — Bedrock model provider; accepts any Bedrock model ID
- `GraphBuilder` — builds the directed acyclic graph (DAG) of agents
- `GraphState` / `Status` — used in conditional edge logic for AND-semantics fan-in

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent import GraphBuilder, Status
from strands.multiagent.graph import GraphState, GraphNode
import logging

logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s")

# Change to DEBUG to see full agent traces (tool calls, API requests, responses)
# Change to WARNING to suppress all but errors
logging.getLogger("strands").setLevel(logging.WARNING)

## 4. Model Configuration

Each `BedrockModel` instance points at a different foundation model. All three workers and the orchestrator use Bedrock — just different `model_id` values.

- **`region_name`** defaults to `AWS_DEFAULT_REGION` env var → `us-west-2` fallback. Override here if needed.
- **`temperature`** controls randomness. Workers use `0.3` for factual consistency; the synthesizer uses `0.2` for tight, deterministic output.
- **`streaming=True`** prevents request timeouts on long responses.

In [4]:
MODEL_IDS = {
    "claude":  "us.anthropic.claude-opus-4-7",           # US cross-region inference profile
    "nova":    "us.amazon.nova-pro-v1:0",                 # US cross-region inference profile
    "mistral": "mistral.mistral-large-3-675b-instruct",   # regional ID (no inference profile)
}

# Claude Opus 4.7: temperature removed (sampling params deprecated)
# Mistral Large 3: plain regional ID, streaming supported
def make_model(key: str, temperature: float = 0.3) -> BedrockModel:
    kwargs = dict(model_id=MODEL_IDS[key], streaming=True)
    if key != "claude":
        kwargs["temperature"] = temperature
    return BedrockModel(**kwargs)

print("Model IDs configured:")
for name, mid in MODEL_IDS.items():
    print(f"  {name:10s} → {mid}")

Model IDs configured:
  claude     → us.anthropic.claude-opus-4-7
  nova       → us.amazon.nova-pro-v1:0
  mistral    → mistral.mistral-large-3-675b-instruct


## 5. Tools — Web Search & Web Fetch

Each worker agent gets two tools so it can actively look up information rather than relying solely on its training data.

### `serpapi_search(query, num_results)`
Calls the [SerpApi](https://serpapi.com) Google Search API and returns organic result titles, URLs, and snippets. Requires `SERPAPI_API_KEY` in your environment (add it to `.env`).

### `web_fetch(url, max_chars)`
Fetches any URL and returns its content as plain text. For HTML pages it strips `<script>`, `<style>`, and all other tags, then decodes HTML entities — leaving just readable text. The `max_chars` cap (default 8000) prevents a single page from consuming the entire context window.

Together these let a worker agent:
1. Search for up-to-date sources on a claim
2. Fetch and read the full content of promising results
3. Cite specific evidence rather than relying on memorised training data

In [5]:
import os
import json
import html
import re
import urllib.request
import urllib.parse
from strands import tool


@tool
def serpapi_search(query: str, num_results: int = 5) -> str:
    """Search the web using SerpApi and return organic search results.

    Args:
        query: The search query string
        num_results: Number of results to return (default 5, max 10)
    """
    api_key = os.environ.get("SERPAPI_API_KEY")
    if not api_key:
        return "Error: SERPAPI_API_KEY not set in environment."

    params = urllib.parse.urlencode({
        "q": query,
        "api_key": api_key,
        "num": min(num_results, 10),
        "output": "json",
    })
    url = f"https://serpapi.com/search.json?{params}"

    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            data = json.loads(resp.read())

        items = []
        for r in data.get("organic_results", [])[:num_results]:
            items.append(
                f"Title: {r.get('title', '')}\n"
                f"URL: {r.get('link', '')}\n"
                f"Snippet: {r.get('snippet', '')}"
            )
        return "\n\n".join(items) if items else "No results found."
    except Exception as e:
        return f"Search error: {e}"


@tool
def web_fetch(url: str, max_chars: int = 8000) -> str:
    """Fetch a web page and return its content as plain text.

    Args:
        url: The URL to fetch
        max_chars: Maximum characters to return (default 8000)
    """
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 (compatible; research-agent/1.0)"},
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            content_type = resp.headers.get("Content-Type", "")
            raw = resp.read().decode("utf-8", errors="replace")

        if "html" in content_type.lower() or raw.lstrip().startswith("<"):
            # Drop script/style blocks, then all tags, then decode entities
            raw = re.sub(r"<(script|style)[^>]*>.*?</\1>", "", raw, flags=re.DOTALL | re.IGNORECASE)
            raw = re.sub(r"<[^>]+>", " ", raw)
            raw = html.unescape(raw)
            raw = re.sub(r"\s+", " ", raw).strip()

        return raw[:max_chars] + ("…" if len(raw) > max_chars else "")
    except Exception as e:
        return f"Fetch error: {e}"


print("Tools defined: serpapi_search, web_fetch")

Tools defined: serpapi_search, web_fetch


## 6. Worker Agents (Fan-out layer)

The three worker agents are structurally identical — same system prompt and tool list, different underlying models. This is intentional: we want the *same question* answered independently by three different model families so their outputs are directly comparable.

Each worker is given two tools and instructed to actively use them:
- **`serpapi_search`** — find current sources
- **`web_fetch`** — read the full content of those sources

This means workers ground their answers in retrieved evidence, not just training-data recall.

In [6]:
WORKER_SYSTEM_PROMPT = """You are a rigorous research assistant. When given a question:

1. Use the serpapi_search tool to find relevant, up-to-date sources.
2. Use the web_fetch tool to read the full content of the most promising results.
3. Provide a thorough, factually grounded answer with specific evidence from what you retrieved.
4. At the end, include a brief section labeled **Confidence & Caveats** where you:
   - Rate your overall confidence (High / Medium / Low)
   - List any claims you are uncertain about
   - Note any areas where your knowledge may be limited or outdated

Be concise but complete. Prefer depth on key points over breadth.
"""

AGENT_TOOLS = [serpapi_search, web_fetch]

claude_worker = Agent(
    name="claude_worker",
    model=make_model("claude"),
    system_prompt=WORKER_SYSTEM_PROMPT,
    tools=AGENT_TOOLS,
)

nova_worker = Agent(
    name="nova_worker",
    model=make_model("nova"),
    system_prompt=WORKER_SYSTEM_PROMPT,
    tools=AGENT_TOOLS,
)

mistral_worker = Agent(
    name="mistral_worker",
    model=make_model("mistral"),
    system_prompt=WORKER_SYSTEM_PROMPT,
    tools=AGENT_TOOLS,
)

print("Worker agents created: claude_worker, nova_worker, mistral_worker")
print("  claude_worker  : us.anthropic.claude-opus-4-7          | streaming=True")
print("  nova_worker    : us.amazon.nova-pro-v1:0                | streaming=True")
print("  mistral_worker : mistral.mistral-large-3-675b-instruct  | streaming=True")

Worker agents created: claude_worker, nova_worker, mistral_worker
  claude_worker  : us.anthropic.claude-opus-4-7          | streaming=True
  nova_worker    : us.amazon.nova-pro-v1:0                | streaming=True
  mistral_worker : mistral.mistral-large-3-675b-instruct  | streaming=True


## 6. Orchestrator Agents (Aggregation & Verification layer)

### 6a. Aggregator

The aggregator receives the original query **plus the outputs of all three workers** automatically (Strands passes upstream node results to downstream nodes in the combined input).

Its job is structured analysis:
- **Consensus points** — facts all three agree on → high confidence
- **Partial agreement** — two of three agree → medium confidence, flag for review
- **Disagreements** — models contradict each other → low confidence, needs verification
- **Unique claims** — only one model mentions something → uncertain, needs verification

The aggregator outputs a structured JSON block that the verifier and synthesizer downstream can parse.

In [7]:
AGGREGATOR_SYSTEM_PROMPT = """\
You are a consensus analyst. You will receive a research question and three independent responses
from different AI models (labeled claude_worker, nova_worker, mistral_worker).

Perform structured agreement analysis and produce output in this exact format:

## CONSENSUS ANALYSIS

### High-Confidence Points (all three agree)
- <point>: <brief explanation of agreement>

### Medium-Confidence Points (two of three agree)
- <point>: <which two agree, what the third says>

### Disputed Claims (models contradict each other)
- <claim>: <model A says X, model B says Y, model C says Z>

### Unique Claims (only one model mentions)
- <claim> [source: <model name>]: <why this might matter>

## VERDICT
CONSENSUS | PARTIAL | DISPUTED
(choose one: CONSENSUS if >80% agreement, DISPUTED if any direct contradictions, PARTIAL otherwise)

## ITEMS REQUIRING VERIFICATION
List specific claims that should be scrutinized further, one per line.
"""

aggregator = Agent(
    name="aggregator",
    model=make_model("claude", temperature=0.1),
    system_prompt=AGGREGATOR_SYSTEM_PROMPT,
)

print("Aggregator agent created")

Aggregator agent created


### 6b. Verifier

The verifier only runs when the aggregator returns `PARTIAL` or `DISPUTED`. It takes disputed/uncertain claims and applies deeper scrutiny:
- Reasons about *why* models might disagree (training cutoff differences, domain ambiguity, genuine factual conflict)
- Weighs which position is better supported by reasoning and evidence
- Produces a resolution recommendation for each disputed point

This is the **tiered verification** step from the architecture: fast consensus check → targeted scrutiny on disputed claims.

In [8]:
VERIFIER_SYSTEM_PROMPT = """\
You are a fact-verification specialist. You will receive:
1. The original research question
2. A consensus analysis identifying disputed or uncertain claims
3. The original responses from all three worker models

For each disputed or uncertain claim:
1. Reason carefully about WHY the models might disagree (training data differences,
   ambiguous phrasing, genuine factual conflict, one model hallucinating)
2. Evaluate the quality of reasoning and evidence each model provides
3. Reach a resolution: ACCEPT one position, REJECT all (if all seem wrong),
   or INCONCLUSIVE (genuinely uncertain)

Output format:

## VERIFICATION REPORT

### Claim: <exact claim text>
**Analysis**: <your reasoning>
**Resolution**: ACCEPT [model name]'s position | REJECT all | INCONCLUSIVE
**Confidence**: High / Medium / Low

(repeat for each disputed claim)

## VERIFIED FACTS
Bullet list of claims that survived verification with their confidence levels.
"""

verifier = Agent(
    name="verifier",
    model=make_model("claude", temperature=0.1),
    system_prompt=VERIFIER_SYSTEM_PROMPT,
)

print("Verifier agent created")

Verifier agent created


### 6c. Synthesizer

The synthesizer is the final node — it produces the answer the user actually sees. It receives everything upstream: the original query, all three worker responses, the aggregation analysis, and (if it ran) the verification report.

It uses **confidence-weighted synthesis**: high-consensus points are stated directly, medium-confidence points are stated with a qualifier, disputed points are either resolved by the verifier or flagged as uncertain.

The output is written for the end user — no internal analysis jargon, just a clear, honest answer.

In [9]:
SYNTHESIZER_SYSTEM_PROMPT = """\
You are a research synthesis specialist. You will receive a research question along with:
- Three independent model responses
- A consensus analysis
- Optionally, a verification report for disputed claims

Produce a single, authoritative final answer that:
1. Leads with the most important, well-supported findings (high-consensus first)
2. Clearly marks lower-confidence claims with phrases like "evidence suggests" or
   "it is likely" rather than stating them as facts
3. Explicitly flags any claims that remain genuinely uncertain after verification
4. Does NOT expose internal analysis details (no mention of model names, voting, etc.)
5. Includes a brief **Confidence Summary** at the end noting overall answer reliability

Write clearly and directly. The reader wants an answer, not a meta-analysis.
"""

synthesizer = Agent(
    name="synthesizer",
    model=make_model("claude", temperature=0.2),
    system_prompt=SYNTHESIZER_SYSTEM_PROMPT,
)

print("Synthesizer agent created")

Synthesizer agent created


## 7. AND-Semantics Fix for Fan-in

Strands Python's `GraphBuilder` re-evaluates conditional edges after every batch of nodes completes, so conditions are **not** permanently dropped on `False`. However, our original condition had a subtle bug:

```python
# ❌ Wrong — compares Status enum across module boundaries
state.results[nid].status == Status.COMPLETED
```

`state.results` is a dict of internal `NodeResult` objects whose `.status` field belongs to a different internal `Status` class than the `Status` we import from `strands.multiagent`. The comparison silently returns `False` even when the node is actually done.

The fix is to use `state.completed_nodes` — the `set[GraphNode]` that the SDK itself maintains authoritatively:

```python
# ✅ Correct — uses the SDK's own completed_nodes set
completed_ids = {n.node_id for n in state.completed_nodes}
return all(nid in completed_ids for nid in node_ids)
```

This is the only change needed. The re-evaluation loop in the SDK works correctly once the condition reads from the right source.

In [10]:
def all_complete(*node_ids: str):
    """AND-semantics gate: True only when every listed node is in completed_nodes."""
    def check(state: GraphState) -> bool:
        # Use state.completed_nodes (set[GraphNode]) — the SDK-authoritative source.
        # Do NOT use state.results[nid].status — that Status comes from a different
        # internal module and silently fails equality checks against our imported Status.
        completed_ids = {n.node_id for n in state.completed_nodes}
        return all(nid in completed_ids for nid in node_ids)
    return check

print("AND-semantics helper defined")

AND-semantics helper defined


## 8. Graph Construction

This cell wires everything together with `GraphBuilder`.

```
claude_worker ──┐
nova_worker   ──┼──(all_complete gate)──► aggregator
mistral_worker  ──┘                              │
                              ┌────────────────┤
                              ▼                ▼
                         synthesizer       verifier
                              ▲                │
                              └────────────────┘
```

**Routing logic:**
- `aggregator → synthesizer`: fires immediately when verdict is `CONSENSUS`
- `aggregator → verifier`: fires when verdict is `PARTIAL` or `DISPUTED`
- `verifier → synthesizer`: always fires after verification completes

In [11]:
def needs_verification(state: GraphState) -> bool:
    """Route to verifier when aggregator finds partial agreement or disputes."""
    agg = next((n for n in state.completed_nodes if n.node_id == "aggregator"), None)
    if not agg:
        return False
    output = str(state.results.get("aggregator", ""))
    return "VERDICT" in output and ("PARTIAL" in output or "DISPUTED" in output)


def is_consensus(state: GraphState) -> bool:
    """Route directly to synthesizer when aggregator finds full consensus."""
    agg = next((n for n in state.completed_nodes if n.node_id == "aggregator"), None)
    if not agg:
        return False
    output = str(state.results.get("aggregator", ""))
    return "VERDICT" in output and "CONSENSUS" in output


def verifier_done(state: GraphState) -> bool:
    """Allow synthesizer to proceed once verifier completes."""
    return any(n.node_id == "verifier" for n in state.completed_nodes)


# Build the graph
builder = GraphBuilder()

builder.add_node(claude_worker, "claude_worker")
builder.add_node(nova_worker,   "nova_worker")
builder.add_node(mistral_worker,  "mistral_worker")
builder.add_node(aggregator,    "aggregator")
builder.add_node(verifier,      "verifier")
builder.add_node(synthesizer,   "synthesizer")

# Fan-out: all three workers are entry points (run simultaneously)
builder.set_entry_point("claude_worker")
builder.set_entry_point("nova_worker")
builder.set_entry_point("mistral_worker")

# Fan-in: aggregator waits for ALL three workers (AND semantics — fixed)
gate = all_complete("claude_worker", "nova_worker", "mistral_worker")
builder.add_edge("claude_worker", "aggregator", condition=gate)
builder.add_edge("nova_worker",   "aggregator", condition=gate)
builder.add_edge("mistral_worker",  "aggregator", condition=gate)

# Routing: consensus → synthesizer directly; disputed → verifier first
builder.add_edge("aggregator", "synthesizer", condition=is_consensus)
builder.add_edge("aggregator", "verifier",    condition=needs_verification)
builder.add_edge("verifier",   "synthesizer", condition=verifier_done)

builder.set_execution_timeout(600)
builder.set_node_timeout(180)
builder.set_max_node_executions(10)

graph = builder.build()
print("Graph built successfully")
print(f"Entry points: {[n.node_id for n in graph.entry_points]}")

Graph built successfully
Entry points: ['nova_worker', 'claude_worker', 'mistral_worker']


## 9. Run the Graph

Submit a research question. The graph fans out to all three workers in parallel, waits for all to complete, then routes through aggregation → optional verification → synthesis.

In [12]:
QUERY = "What are the main causes of the 2008 financial crisis and what reforms were enacted to prevent a recurrence?"

print(f"Query: {QUERY}")
print("-" * 60)
print("Running graph...\n")

result = graph(QUERY)

print(f"\nGraph status     : {result.status}")
print(f"Completed nodes  : {result.completed_nodes}")
print(f"Failed nodes     : {result.failed_nodes}")
print(f"Total time       : {result.execution_time/1000:.1f}s")
print(f"Execution order  : {[n.node_id for n in result.execution_order]}")

Query: What are the main causes of the 2008 financial crisis and what reforms were enacted to prevent a recurrence?
------------------------------------------------------------
Running graph...

<thinking> To answer this question, I need to identify the main causes of the 2008 financial crisis and the reforms enacted to prevent a recurrence. I will start by using the serpapi_search tool to find relevant sources on the causes of the 2008 financial crisis. Then, I will use the web_fetch tool to read the full content of the most promising results.I'll research this question to give you a well-grounded answer on After that, I will repeat the process to find information on the reforms enacted. </thinking>


Tool #1: serpapi_search
 both the causes and the regulatory response.
Tool #1: serpapi_search

Tool #2: serpapi_search

Tool #1: serpapi_search
<thinking> The search results provide a good overview of the main causes of the 2008 financial crisis
Tool #2: serpapi_search
. I will now use t

In [13]:
def diagnose(result):
    """Print per-node status, timing, and error details."""
    print(f"Overall status : {result.status}")
    print(f"Execution time : {result.execution_time/1000:.1f}s")
    print(f"Execution order: {[n.node_id for n in result.execution_order]}")
    print()

    all_node_ids = ["claude_worker", "nova_worker", "mistral_worker",
                    "aggregator", "verifier", "synthesizer"]

    for node_id in all_node_ids:
        node_result = result.results.get(node_id)
        if node_result is None:
            print(f"  - [{node_id:<20}]  DID NOT RUN")
            continue

        status = node_result.status
        marker = "✓" if str(status).endswith("COMPLETED") else "✗"
        print(f"  {marker} [{node_id:<20}]  {status}")

        err = node_result.result
        if not str(status).endswith("COMPLETED"):
            if isinstance(err, Exception):
                import traceback
                print(f"      Exception : {type(err).__name__}: {err}")
                tb = getattr(err, '__traceback__', None)
                if tb:
                    for line in traceback.format_tb(tb):
                        print("        " + line.rstrip())
            else:
                print(f"      Detail    : {str(err)[:200]}")
        else:
            print(f"      Preview   : {str(err)[:150].replace(chr(10), ' ')}…")
        print()

diagnose(result)

Overall status : Status.COMPLETED
Execution time : 0.0s
Execution order: ['nova_worker', 'mistral_worker', 'claude_worker', 'aggregator', 'synthesizer']

  ✓ [claude_worker       ]  Status.COMPLETED
      Preview   : I have comprehensive material. Let me synthesize a thorough answer.  # Causes of the 2008 Financial Crisis and Reforms Enacted to Prevent Recurrence  …

  ✓ [nova_worker         ]  Status.COMPLETED
      Preview   : The main causes of the 2008 financial crisis were:  1. **Subprime Lending**: Excessive speculation on property values by both homeowners and financial…

  ✓ [mistral_worker      ]  Status.COMPLETED
      Preview   : ### **Main Causes of the 2008 Financial Crisis**  The 2008 financial crisis was the result of a complex interplay of factors, including financial inno…

  ✓ [aggregator          ]  Status.COMPLETED
      Preview   : ## CONSENSUS ANALYSIS  ### High-Confidence Points (all three agree) - **Housing bubble and subprime lending**: All three identify the U

## 10. Inspect Results

Each node's output is stored in `result.results[node_id].result`.

In [14]:
def show(node_id: str, label: str = None):
    node_result = result.results.get(node_id)
    if node_result is None:
        print(f"[{node_id}] did not run")
        return
    title = label or node_id
    print(f"{'='*60}")
    print(f"  {title}  (status: {node_result.status})")
    print(f"{'='*60}")
    print(str(node_result.result))
    print()

In [15]:
show("claude_worker",  "Claude Worker Response")
show("nova_worker",    "Nova Worker Response")
show("mistral_worker", "Mistral Worker Response")

  Claude Worker Response  (status: Status.COMPLETED)
I have comprehensive material. Let me synthesize a thorough answer.

# Causes of the 2008 Financial Crisis and Reforms Enacted to Prevent Recurrence

## Main Causes

The 2008 financial crisis (also called the Global Financial Crisis) arose from a combination of interconnected factors that built up over the preceding decade:

### 1. The U.S. Housing Bubble and Subprime Lending
- Average U.S. home prices **more than doubled between 1998 and 2006** — the sharpest increase in U.S. history (Federal Reserve History).
- Home ownership rose from 64% (1994) to 69% (2005), and mortgage debt jumped from **61% of GDP in 1998 to 97% in 2006**.
- Lenders aggressively extended **subprime mortgages** to borrowers with poor credit or little documentation, often with adjustable rates ("teaser" rates that reset higher).

### 2. Securitization and Complex Financial Products
- Banks bundled mortgages (including subprime ones) into **mortgage-backed secur

In [16]:
show("aggregator", "Consensus Analysis")

  Consensus Analysis  (status: Status.COMPLETED)
## CONSENSUS ANALYSIS

### High-Confidence Points (all three agree)
- **Housing bubble and subprime lending**: All three identify the U.S. housing bubble, doubling of home prices (1998-2006), and subprime mortgage lending as primary causes.
- **Securitization/complex financial products**: All cite MBS and CDOs as mechanisms that spread risk globally while weakening lending standards.
- **Excessive leverage**: All agree financial institutions (notably investment banks) operated with dangerously high leverage ratios.
- **Deregulation/regulatory gaps**: All cite the 1999 repeal of Glass-Steagall (Gramm-Leach-Bliley) as a contributing factor.
- **Shadow banking system**: All three identify unregulated non-bank financial institutions as a key vulnerability.
- **Dodd-Frank Act (2010)** as the central U.S. reform, including: Volcker Rule, CFPB, FSOC, Orderly Liquidation Authority, derivatives regulation, and stress testing.
- **Basel III** as t

In [17]:
show("verifier", "Verification Report")

[verifier] did not run


In [18]:
show("synthesizer", "Final Answer")

  Final Answer  (status: Status.COMPLETED)
# The 2008 Financial Crisis: Causes and Reforms

## Main Causes

The 2008 financial crisis resulted from a convergence of interrelated factors rather than any single cause.

### Housing Bubble and Subprime Lending
U.S. home prices roughly doubled between the late 1990s and 2006, fueled by aggressive subprime mortgage lending to borrowers with weak credit. Many loans featured adjustable rates, minimal documentation, and low or no down payments. When home prices stalled and then fell sharply (the Case-Shiller national index declined roughly 27% from its 2006 peak to its 2012 trough), defaults cascaded through the system.

### Securitization and Complex Financial Products
Mortgages were bundled into mortgage-backed securities (MBS) and collateralized debt obligations (CDOs), which were sold globally. This "originate-to-distribute" model weakened lending standards because originators bore little long-term risk. Credit default swaps (CDS)—most noto

## 11. Token Usage Summary

The graph accumulates token usage across all nodes. This cell shows cost transparency: how many tokens each node consumed.

In [19]:
print(f"{'Node':<20} {'Input':>10} {'Output':>10} {'Total':>10}")
print("-" * 54)
for node_id, node_result in result.results.items():
    usage = getattr(node_result, "accumulated_usage", {})
    if usage:
        inp = usage.get("inputTokens", 0)
        out = usage.get("outputTokens", 0)
        tot = usage.get("totalTokens", inp + out)
        print(f"{node_id:<20} {inp:>10,} {out:>10,} {tot:>10,}")

total = result.accumulated_usage
if total:
    print("-" * 54)
    print(f"{'TOTAL':<20} {total.get('inputTokens',0):>10,} {total.get('outputTokens',0):>10,} {total.get('totalTokens',0):>10,}")

Node                      Input     Output      Total
------------------------------------------------------
nova_worker              14,477      1,181     15,658
mistral_worker           13,453      3,855     17,308
claude_worker            18,195      3,009     21,204
aggregator                9,844      1,850     11,694
synthesizer               2,186      2,555      4,741
------------------------------------------------------
TOTAL                    58,155     12,450     70,605


## 12. Reusable Query Function

For convenience, wrap the graph execution in a function you can call repeatedly without rebuilding the graph.

In [21]:
def research(question: str, verbose: bool = False) -> str:
    """Run a question through the three-agent research graph and return the final answer."""
    res = graph(question)

    if verbose:
        diagnose(res)

    synth = res.results.get("synthesizer")
    if synth and str(synth.status).endswith("COMPLETED"):
        return str(synth.result)

    agg = res.results.get("aggregator")
    if agg:
        return str(agg.result)

    return f"Graph completed with status {res.status} but no synthesized output."


# Quick test
answer = research(
    "How does AWS makes sure AWS employees including service operators cannot access to customer data?",
    verbose=True,
)
print(answer)


I'll research AAWS employs a variety of measures to ensure that its employees, including service operators, cannot access customer data without properWS's specific controls for preventing employee access to customer data.
Tool #8: serpapi_search

Tool #9: serpapi_search
 authorization. Here are some of the key mechanisms in place:

### 1. **Physical Security**

Tool #10: serpapi_search
- **Data Centers**: AWS data centers are highly secured facilities with multiple layers of physical security, including
Tool #6: serpapi_search
 biometric scans, security guards, and surveillance cameras.
- **Access Controls**: Only authorized personnel can enter data centers, and access is tightly controlled and monitored.

### 2. **Logical Security**
-
Tool #7: serpapi_search
 **IAM (Identity and Access Management)**: AWS uses IAM to manage access to AWS services and resources securely. Employees and operators must use IAM roles and permissions to access any resources.
- **Least Privilege Principle**: 

---

## Extending This System

**Swap models:** Change `MODEL_IDS` in Section 4. Any Bedrock model ID works — GTP-5.5 once become available.

**Persist results:** The `result.results` dict is plain Python — serialize to JSON with `json.dumps({k: str(v.result) for k, v in result.results.items()})`.

**Add a debate round:** For `DISPUTED` verdicts, add a second aggregator node that receives the verifier output and re-runs a focused debate between workers on just the disputed claims. Wire it with another conditional edge from `verifier`.

**Amazon Bedrock AgentCore:** To accelerate agents to production with composable services